# 19.10 如何把成品帶到較小的電腦？


先問電腦要放下哪些數字。MoE 每個 token 只使用部分 expert，下一個 token 卻可能選另一組，因此本實作仍把所有 expert 的權重載入記憶體。少用一部分計算，並沒有把其他權重從 RAM 移走。

這份 MoE 共有 **5,447,107 個參數，約5.447M**；Dense 共有 **2,288,067 個，約2.288M**，兩者都包括文字、影像、OCR與語音元件。M在此表示一百萬個數字。公開版以 FP32 計算，每個參數4 bytes；先估數值本身的儲存：


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
for name, count in [("MoE", 5_447_107), ("Dense", 2_288_067)]:
    weight_bytes = count * 4
    print(name, "FP32權重bytes", weight_bytes, "約MiB", round(weight_bytes / 1024**2, 2))

結果約為 MoE 20.78 MiB、Dense 8.73 MiB，MiB以1024² bytes計。這只計每個可學參數一次；公開檔保存張量的安排與格式資訊也占空間，實際檔案大小要另核對。載入時也會有暫存副本，執行另需模態特徵、KV cache與工作空間。它不是整個程式的 RAM 峰值，更不能直接當成電腦最低需求。

若資源仍不足，可以沿用[第17章的量化](https://birdhackor.github.io/tiny-perceptron-vlm/chapter-17.html)或[第18章的蒸餾](https://birdhackor.github.io/tiny-perceptron-vlm/chapter-18.html)思路。量化以較少位元近似保存同一結構的權重；蒸餾先選較小學生，再向教師的文字或分布學習。這兩章已教過機制，但**本輪 V2 公開交付的是原浮點模型，沒有完成並驗收這份最終成品的全量量化版或蒸餾版**。

選部署版本時，應把三個問題並排：檔案多大、載入與生成占多少資源、同一任務的答案保留多少。Dense 的參數較少，最終能力也有自己的結果，見[19.12](https://birdhackor.github.io/tiny-perceptron-vlm/19.12.html)；不能把它改稱 MoE 的蒸餾學生。兩條路線的訓練目標與歷程不同，這份比較也不能單獨證明架構造成某項差異。

練習另算「每個數字2 bytes」的理想數值儲存，預測兩個合計都減半。這一步只改算術；要採用實際低精度或量化版本，仍須轉換、存檔、重載，並重新核對任務與資源。
